In [7]:
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download("punkt")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

***1. Preprocessing (NO Stopwords)***

In [8]:
def preprocess(text):

    tokens = word_tokenize(text.lower())

    lemmatizer = WordNetLemmatizer()

    tokens = [
        lemmatizer.lemmatize(w)
        for w in tokens
        if w.isalpha()
    ]

    return tokens

***2. Vocabulary***

In [9]:
def build_vocab(tokens):

    vocab = list(set(tokens))

    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for i, w in enumerate(vocab)}

    return vocab, word2idx, idx2word

***3. One Hot Encoding***

In [10]:
def one_hot(word, word2idx, size):

    vec = np.zeros(size)

    if word in word2idx:
        vec[word2idx[word]] = 1

    return vec

***4. Create CBOW Data***

In [11]:
def create_data(tokens, window=2):

    X, Y = [], []

    for i in range(window, len(tokens) - window):

        context = []

        for j in range(i - window, i + window + 1):

            if j != i:
                context.append(tokens[j])

        X.append(context)
        Y.append(tokens[i])

    return X, Y

***5. Softmax***

In [12]:
def softmax(x):

    e = np.exp(x - np.max(x))
    return e / np.sum(e)

***6. Forward Propagation***

In [13]:
def forward(x, W1, W2):

    h = np.dot(x, W1)      # Hidden = Embedding
    y = np.dot(h, W2)     # Output

    return h, softmax(y)

***7. Backward Propagation***

In [14]:
def backward(x, h, y, t, W2):

    error = y - t

    dW2 = np.outer(h, error)

    dW1 = np.outer(x, np.dot(W2, error))

    return dW1, dW2

***8. Train CBOW***

In [15]:
def train(X, Y, word2idx, size, embed=10, lr=0.01, epochs=500):

    W1 = np.random.rand(size, embed)   # Embedding Matrix
    W2 = np.random.rand(embed, size)

    for e in range(epochs):

        loss = 0

        for context, target in zip(X, Y):

            x = np.mean([
                one_hot(w, word2idx, size)
                for w in context
            ], axis=0)

            t = one_hot(target, word2idx, size)

            h, y = forward(x, W1, W2)

            loss += -np.sum(t * np.log(y + 1e-9))

            dW1, dW2 = backward(x, h, y, t, W2)

            W1 -= lr * dW1
            W2 -= lr * dW2

        if e % 100 == 0:
            print("Epoch:", e, "Loss:", round(loss, 3))

    return W1, W2

***9. Get Word Embeddings***

In [16]:
def get_embeddings(W1, vocab):

    embeddings = {}

    for i, word in enumerate(vocab):

        embeddings[word] = W1[i]

    return embeddings

***10. Prediction***

In [17]:
def predict(context, W1, W2, word2idx, idx2word, size):

    x = np.mean([
        one_hot(w, word2idx, size)
        for w in context
    ], axis=0)

    _, y = forward(x, W1, W2)

    return idx2word[np.argmax(y)]

***11. Main Function***

In [18]:
def main():

    corpus = """
    Virat Kohli is a great cricket player.
    He scores many runs in every match.
    MS Dhoni is a famous captain.
    India won many cricket matches.
    Football players train hard every day.
    Messi scores beautiful goals.
    Ronaldo is a strong striker.
    """

    # Preprocess
    tokens = preprocess(corpus)

    print("Tokens:", tokens)

    # Vocabulary
    vocab, word2idx, idx2word = build_vocab(tokens)
    size = len(vocab)

    # CBOW Data
    X, Y = create_data(tokens)

    # Train
    W1, W2 = train(X, Y, word2idx, size)

    # Get embeddings
    embeddings = get_embeddings(W1, vocab)

    print("\nWord Embedding Example:")
    print("cricket →", embeddings["cricket"])

    # Test prediction
    test = ["virat", "kohli", "great", "cricket"]

    result = predict(test, W1, W2, word2idx, idx2word, size)

    print("\nContext:", test)
    print("Predicted Word:", result)

In [19]:
if __name__ == "__main__":
    main()

Tokens: ['virat', 'kohli', 'is', 'a', 'great', 'cricket', 'player', 'he', 'score', 'many', 'run', 'in', 'every', 'match', 'm', 'dhoni', 'is', 'a', 'famous', 'captain', 'india', 'won', 'many', 'cricket', 'match', 'football', 'player', 'train', 'hard', 'every', 'day', 'messi', 'score', 'beautiful', 'goal', 'ronaldo', 'is', 'a', 'strong', 'striker']
Epoch: 0 Loss: 128.662
Epoch: 100 Loss: 100.532
Epoch: 200 Loss: 73.348
Epoch: 300 Loss: 43.656
Epoch: 400 Loss: 22.834

Word Embedding Example:
cricket → [-0.23653402  0.69109937 -0.17216045  2.89619928 -0.00906911 -0.10121876
  0.76348395  1.21805042  1.91590538 -0.86813861]

Context: ['virat', 'kohli', 'great', 'cricket']
Predicted Word: player
